# 1. Data Entry

For my data, I used [this](https://www.kaggle.com/datasets/mdabbert/ultimate-ufc-dataset) kaggle dataset from mdabbert. They were able to scrape [ufcstats.com](ufcstats.com) for fighter and bout information, [bestfightodds.com](bestfightodds.com) for closing odds, and [a separate kaggle dataset](https://www.kaggle.com/datasets/martj42/ufc-rankings) for historical ranking information. In this notebook, we are just going to look for missing data or potentially incorrect information.

In [1]:
# We don't need many imports for this step
import pandas as pd
pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', None) 

In [2]:
# Bringing in our csv data
raw_df = pd.read_csv("../data/ufc-master.csv")

In [3]:
# Taking a look at all of our columns

colCount = 0
for each in raw_df:
    colCount += 1
    print(str(colCount) + " " + each)

1 RedFighter
2 BlueFighter
3 RedOdds
4 BlueOdds
5 RedExpectedValue
6 BlueExpectedValue
7 Date
8 Location
9 Country
10 Winner
11 TitleBout
12 WeightClass
13 Gender
14 NumberOfRounds
15 BlueCurrentLoseStreak
16 BlueCurrentWinStreak
17 BlueDraws
18 BlueAvgSigStrLanded
19 BlueAvgSigStrPct
20 BlueAvgSubAtt
21 BlueAvgTDLanded
22 BlueAvgTDPct
23 BlueLongestWinStreak
24 BlueLosses
25 BlueTotalRoundsFought
26 BlueTotalTitleBouts
27 BlueWinsByDecisionMajority
28 BlueWinsByDecisionSplit
29 BlueWinsByDecisionUnanimous
30 BlueWinsByKO
31 BlueWinsBySubmission
32 BlueWinsByTKODoctorStoppage
33 BlueWins
34 BlueStance
35 BlueHeightCms
36 BlueReachCms
37 BlueWeightLbs
38 RedCurrentLoseStreak
39 RedCurrentWinStreak
40 RedDraws
41 RedAvgSigStrLanded
42 RedAvgSigStrPct
43 RedAvgSubAtt
44 RedAvgTDLanded
45 RedAvgTDPct
46 RedLongestWinStreak
47 RedLosses
48 RedTotalRoundsFought
49 RedTotalTitleBouts
50 RedWinsByDecisionMajority
51 RedWinsByDecisionSplit
52 RedWinsByDecisionUnanimous
53 RedWinsByKO
54 RedWins

The first thing I noticed here is how many different columns there are! The creators of this dataset did an amazing job getting everything that's available, I just don't think I want all of this. Overtraining the model with so many data types might confuse it. I'm going to choose some columns that I'd like to try training models with later on.

I'm going a few different ways here. I want some that describe the fighters as old or young, streaking or slumping, strikers or grapplers, seasoned or green, and even tall or short. At this point, I can see myself trying out a few different kinds of models and I want to set myself up for success later on.

In [4]:
# Making a list of useful columns and creating a new dataframe with only those columns included
useful_columns = ['RedFighter', 'BlueFighter', 'Finish', 'FinishRound', 'RedOdds', 'BlueOdds',
    'Date', 'Winner', 'TitleBout', 'WeightClass', 'Gender', 'NumberOfRounds', 
    'BlueCurrentLoseStreak', 'BlueCurrentWinStreak', 'BlueLongestWinStreak', 'BlueLosses', 'BlueTotalRoundsFought', 'BlueTotalTitleBouts', 
    'BlueWinsByKO', 'BlueWinsBySubmission', 'BlueWins', 'BlueStance', 'BlueHeightCms', 'BlueReachCms', 'BlueAge', 'BMatchWCRank', 
    'RedCurrentLoseStreak', 'RedCurrentWinStreak', 'RedLongestWinStreak', 'RedLosses', 'RedTotalRoundsFought', 'RedTotalTitleBouts', 
    'RedWinsByKO', 'RedWinsBySubmission', 'RedWins', 'RedStance', 'RedHeightCms', 'RedReachCms', 'RedAge', 'RMatchWCRank', 
    'BetterRank']

dfColumns = raw_df[useful_columns].copy()

Now that we have that done, let's check where we're missing values. 

In [5]:
# Creating a new table that displays how many missing cells we have and how much of the column that takes up 

missing = pd.concat([dfColumns.isnull().sum(), 100 * dfColumns.isnull().mean()], axis=1)
missing.columns=['count', '%']
missing.sort_values(by='count', ascending = False).head(10)

,count,%
BMatchWCRank,5328,81.617647
RMatchWCRank,4749,72.748162
FinishRound,622,9.528186
Finish,238,3.645833
RedOdds,227,3.477328
BlueOdds,226,3.462010
BlueStance,3,0.045956
RedFighter,0,0.000000
RedTotalTitleBouts,0,0.000000
RedCurrentLoseStreak,0,0.000000


As we can see, there are 6 columns with missing values: Rankings for each fighter, fight odds for each fighter, finish type and finish round. I'm not worried about the ranking because most fighters are not ranked, and I'll just fill that in later. I do want to see why some odds and finish descriptions aren't included because ~3.5% of fighters' odds is a significant portion of the dataset and finishing information is even higher.

In [6]:
# Creating a nullcheck table to see missing finishing information

nullCheck = dfColumns[dfColumns['FinishRound'].isna()][['Finish', 'FinishRound', 'Date', 'RedFighter', 'BlueFighter']]
print(nullCheck.head(10))

      Finish  FinishRound        Date       RedFighter            BlueFighter
1807   U-DEC          NaN  2021-05-22         Rob Font         Cody Garbrandt
1808  KO/TKO          NaN  2021-05-22      Yan Xiaonan          Carla Esparza
1809   U-DEC          NaN  2021-05-22      Justin Tafa         Jared Vanderaa
1810   S-DEC          NaN  2021-05-22  Felicia Spencer           Norma Dumont
1811   U-DEC          NaN  2021-05-22    Ricardo Ramos             Bill Algeo
1812   U-DEC          NaN  2021-05-22  Jack Hermansson       Edmen Shahbazyan
1813     SUB          NaN  2021-05-22     Ben Rothwell          Chris Barnett
1814   U-DEC          NaN  2021-05-22      Court McGee          Claudio Silva
1815  KO/TKO          NaN  2021-05-22      Bruno Silva       Victor Rodriguez
1816   U-DEC          NaN  2021-05-22     Josh Culibao  Shayilan Nuerdanbieke


In [7]:
# Creating a nullcheck table to see missing betting line information

nullCheck = dfColumns[dfColumns['RedOdds'].isna()][['RedOdds', 'BlueOdds', 'Date', 'RedFighter', 'BlueFighter']]
print(nullCheck.head(10))

    RedOdds  BlueOdds        Date                 RedFighter  \
16      NaN    -230.0  2024-11-23                 Song Kenan   
38      NaN       NaN  2024-11-16             Veronica Hardy   
44      NaN       NaN  2024-11-09      Karolina Kowalkiewicz   
45      NaN       NaN  2024-11-09  Elizeu Zaleski dos Santos   
46      NaN       NaN  2024-11-09       Matthew Semelsberger   
47      NaN       NaN  2024-11-09               Cody Stamann   
48      NaN       NaN  2024-11-09               Tresean Gore   
49      NaN       NaN  2024-11-09            Melissa Mullins   
56      NaN       NaN  2024-11-02             Aiemann Zahabi   
57      NaN       NaN  2024-11-02           Charles Jourdain   

          BlueFighter  
16    Muslim Salikhov  
38      Eduarda Moura  
44       Denise Gomes  
45   Zachary Scroggin  
46     Charles Radtke  
47  Da'Mon Blackshear  
48    Antonio Trocoli  
49     Klaudia Sygula  
56       Pedro Munhoz  
57       Victor Henry  


There seem to be some events where the scraper had trouble getting betting or finish information. Let me bring this into excel and see what I can do.

In [8]:
# Saving the shortened data to csv

dfColumns.to_csv('../data/df_columns.csv')

# 2. Data Cleaning

So after looking at the data, some data wasn't able to be scraped because of the sites used for betting or finishing information and the way they're organized. This is something I will need to fix myself before I launch the model. For now, I manually fixed some of the data so we had enough historical information as possible.

In [9]:
# Bringing in the data I fixed and checking the missing value numbers

dfFilled = pd.read_csv('../data/df_finishes.csv')

missing = pd.concat([dfFilled.isnull().sum(), 100 * dfFilled.isnull().mean()], axis=1)
missing.columns=['count', '%']
missing.sort_values(by='count', ascending = False).head(10)

,count,%
BMatchWCRank,5339,81.623605
RMatchWCRank,4760,72.771747
FinishRound,158,2.415533
Finish,157,2.400245
BlueStance,3,0.045865
Unnamed: 0,0,0.000000
RedTotalRoundsFought,0,0.000000
BlueReachCms,0,0.000000
BlueAge,0,0.000000
RedCurrentLoseStreak,0,0.000000


It looks like we're down to only about 2.4% missing finishing information and no missing betting information. I was able to find patterns of events where data was missing and quickly fill that in, but the remaining would take too long to fix, so I will take the 2.4% hit here. Let's keep cleaning this data up, including that weird Unnamed:0 column that appeared when I sent the data to csv and forgot to write index=False.

In [10]:
# Checking whose stance information is missing

nullCheck = dfFilled[dfFilled['BlueStance'].isna()][['BlueFighter', 'BlueStance']]
print(nullCheck.head(10))

              BlueFighter BlueStance
401   Daria Zhelezniakova        NaN
1830  Juancamilo Ronderos        NaN
1881          Juan Espino        NaN


In [11]:
# A quick search helped me fill this in

dfFilled.loc[dfFilled['BlueFighter'] == 'Daria Zhelezniakova', 'BlueStance'] = 'Orthodox'
dfFilled.loc[dfFilled['BlueFighter'] == 'Juancamilo Ronderos', 'BlueStance'] = 'Southpaw'
dfFilled.loc[dfFilled['BlueFighter'] == 'Juan Espino', 'BlueStance'] = 'Orthodox'

nullCheck = dfFilled[dfFilled['BlueStance'].isna()][['BlueFighter', 'BlueStance']]
print(nullCheck.head(10))

Empty DataFrame
Columns: [BlueFighter, BlueStance]
Index: []


In [12]:
# Dropping rows with unfilled finishing information

df = dfFilled.dropna(subset=['Finish', 'FinishRound']).copy()
df = df.drop('Unnamed: 0', axis=1)

# Filling the Unranked fighters' rank cells with 20
df[['BMatchWCRank', 'RMatchWCRank']] = df[['BMatchWCRank', 'RMatchWCRank']].fillna(20)

# Changing the date dtype
df['Date'] = pd.to_datetime(df['Date'], format="%m/%d/%Y")
print(df['Date'].dtype)

datetime64[ns]


Above we inputted the missing fighters' stances and dropped rows where finishing information wasn't included. I put 20 as the fill rank for unranked fighters to show that a fighter unranked in a weightclass would typically be below a ranked fighter.

I, like others, believe that MMA has evolved a lot over the years. There is fight data in the original dataset that doesn't include rankings not because the fighters weren't ranked, but because rankings weren't introduced until 2013. I want to remove some fights eventually, but for the sake of having a more complete database, I won't drop any yet.

In [13]:
#Final missing data check
missing = pd.concat([df.isnull().sum(), 100 * df.isnull().mean()], axis=1)
missing.columns=['count', '%']
missing.sort_values(by='count', ascending = False).head()

,count,%
RedFighter,0,0.0
BlueStance,0,0.0
BlueReachCms,0,0.0
BlueAge,0,0.0
BMatchWCRank,0,0.0


In [14]:
# Checking how many fights we have

print(len(df))

6383


In [15]:
# Looking at our data

df.head()

,RedFighter,BlueFighter,Finish,FinishRound,RedOdds,BlueOdds,Date,Winner,TitleBout,WeightClass,Gender,NumberOfRounds,BlueCurrentLoseStreak,BlueCurrentWinStreak,BlueLongestWinStreak,BlueLosses,BlueTotalRoundsFought,BlueTotalTitleBouts,BlueWinsByKO,BlueWinsBySubmission,BlueWins,BlueStance,BlueHeightCms,BlueReachCms,BlueAge,BMatchWCRank,RedCurrentLoseStreak,RedCurrentWinStreak,RedLongestWinStreak,RedLosses,RedTotalRoundsFought,RedTotalTitleBouts,RedWinsByKO,RedWinsBySubmission,RedWins,RedStance,RedHeightCms,RedReachCms,RedAge,RMatchWCRank,BetterRank
0,Colby Covington,Joaquin Buckley,KO/TKO,3.0,205,-250,2024-12-14,Blue,False,Welterweight,MALE,5,0,5,5,4,34,0,7,0,10,Southpaw,177.80,193.04,30,9.0,1,0,7,4,58,4,3,2,12,Orthodox,180.34,182.88,36,6.0,Red
1,Cub Swanson,Billy Quarantillo,KO/TKO,3.0,124,-148,2024-12-14,Red,False,Featherweight,MALE,3,1,0,4,4,28,0,4,1,7,Orthodox,177.80,177.80,36,20.0,1,0,6,13,82,0,6,2,19,Orthodox,172.72,177.80,41,20.0,neither
2,Manel Kape,Bruno Silva,KO/TKO,3.0,-395,310,2024-12-14,Red,False,Flyweight,MALE,3,0,4,4,2,16,0,3,1,4,Orthodox,162.56,165.10,34,12.0,1,0,4,3,17,0,2,0,4,Southpaw,165.10,172.72,31,9.0,Red
3,Vitor Petrino,Dustin Jacoby,KO/TKO,3.0,-340,270,2024-12-14,Blue,False,Light Heavyweight,MALE,3,2,0,4,6,35,0,4,0,8,Orthodox,190.50,193.04,36,20.0,1,0,5,1,14,0,2,1,5,Orthodox,187.96,195.58,27,20.0,neither
4,Adrian Yanez,Daniel Marcos,S-DEC,3.0,185,-225,2024-12-14,Blue,False,Bantamweight,MALE,3,0,4,4,0,13,0,1,0,4,Orthodox,170.18,175.26,31,20.0,0,1,6,2,15,0,6,0,7,Orthodox,170.18,177.80,31,20.0,neither


Just taking a look at the data here, I want to clean it up a little bit. We have some floats where we don't need them and I'd like the objects that say corner colors to be boolean to help the computer out with the predictions.

In [16]:
# Changing some floats to integers

to_int = ['FinishRound', 'BMatchWCRank', 'RMatchWCRank']
df[to_int] = df[to_int].astype('int64')

# Making our winner column boolean to make things easier for the model

df['RedWinner'] = df['Winner'].map({'Red': True, 'Blue': False}).astype('bool')
df = df.drop('Winner', axis=1)
df = df.drop('BetterRank', axis=1)

# Double checking

df.dtypes

RedFighter                       object
BlueFighter                      object
Finish                           object
FinishRound                       int64
RedOdds                           int64
BlueOdds                          int64
Date                     datetime64[ns]
TitleBout                          bool
WeightClass                      object
Gender                           object
NumberOfRounds                    int64
BlueCurrentLoseStreak             int64
BlueCurrentWinStreak              int64
BlueLongestWinStreak              int64
BlueLosses                        int64
BlueTotalRoundsFought             int64
BlueTotalTitleBouts               int64
BlueWinsByKO                      int64
BlueWinsBySubmission              int64
BlueWins                          int64
BlueStance                       object
BlueHeightCms                   float64
BlueReachCms                    float64
BlueAge                           int64
BMatchWCRank                      int64


Now that I removed the Winner column and instead made it RedWinner, we are going to be looking for results on a True/False basis instead of a Red/Blue basis. Since I have this, I want to add a couple of columns that deal specifically with the return for each bet. It will be a scale you multiply the amount of money bet, and it will show you how much your bet would return.

In [17]:
# Defining my function

def calculate_return(odds, won):
    if not won:
        return 0
    if odds < 0:
        return round(1 + (-100 / odds), 4)
    else:
        return round(1 + (odds / 100), 4)

# Creating 2 new columns with the funtion appropriately applied    
    
df['RedReturn'] = df.apply(lambda row: calculate_return(row['RedOdds'], row['RedWinner']), axis=1)
df['BlueReturn'] = df.apply(lambda row: calculate_return(row['BlueOdds'], not row['RedWinner']), axis=1)

# Putting my columns in a different order for better readability

columns = ['RedFighter','BlueFighter','RedOdds','BlueOdds','RedWinner','RedReturn','BlueReturn',
           'Date','TitleBout','WeightClass','Gender','NumberOfRounds','Finish','FinishRound',
           'BlueCurrentLoseStreak','BlueCurrentWinStreak','BlueLongestWinStreak','BlueLosses','BlueTotalRoundsFought',
           'BlueTotalTitleBouts','BlueWinsByKO','BlueWinsBySubmission','BlueWins','BlueStance','BlueHeightCms','BlueReachCms',
           'BlueAge','BMatchWCRank', 
           'RedCurrentLoseStreak','RedCurrentWinStreak','RedLongestWinStreak','RedLosses','RedTotalRoundsFought',
           'RedTotalTitleBouts','RedWinsByKO','RedWinsBySubmission','RedWins','RedStance','RedHeightCms','RedReachCms',
           'RedAge','RMatchWCRank',]

df = df[columns]

df.head()

,RedFighter,BlueFighter,RedOdds,BlueOdds,RedWinner,RedReturn,BlueReturn,Date,TitleBout,WeightClass,Gender,NumberOfRounds,Finish,FinishRound,BlueCurrentLoseStreak,BlueCurrentWinStreak,BlueLongestWinStreak,BlueLosses,BlueTotalRoundsFought,BlueTotalTitleBouts,BlueWinsByKO,BlueWinsBySubmission,BlueWins,BlueStance,BlueHeightCms,BlueReachCms,BlueAge,BMatchWCRank,RedCurrentLoseStreak,RedCurrentWinStreak,RedLongestWinStreak,RedLosses,RedTotalRoundsFought,RedTotalTitleBouts,RedWinsByKO,RedWinsBySubmission,RedWins,RedStance,RedHeightCms,RedReachCms,RedAge,RMatchWCRank
0,Colby Covington,Joaquin Buckley,205,-250,False,0.0000,1.4000,2024-12-14,False,Welterweight,MALE,5,KO/TKO,3,0,5,5,4,34,0,7,0,10,Southpaw,177.80,193.04,30,9,1,0,7,4,58,4,3,2,12,Orthodox,180.34,182.88,36,6
1,Cub Swanson,Billy Quarantillo,124,-148,True,2.2400,0.0000,2024-12-14,False,Featherweight,MALE,3,KO/TKO,3,1,0,4,4,28,0,4,1,7,Orthodox,177.80,177.80,36,20,1,0,6,13,82,0,6,2,19,Orthodox,172.72,177.80,41,20
2,Manel Kape,Bruno Silva,-395,310,True,1.2532,0.0000,2024-12-14,False,Flyweight,MALE,3,KO/TKO,3,0,4,4,2,16,0,3,1,4,Orthodox,162.56,165.10,34,12,1,0,4,3,17,0,2,0,4,Southpaw,165.10,172.72,31,9
3,Vitor Petrino,Dustin Jacoby,-340,270,False,0.0000,3.7000,2024-12-14,False,Light Heavyweight,MALE,3,KO/TKO,3,2,0,4,6,35,0,4,0,8,Orthodox,190.50,193.04,36,20,1,0,5,1,14,0,2,1,5,Orthodox,187.96,195.58,27,20
4,Adrian Yanez,Daniel Marcos,185,-225,False,0.0000,1.4444,2024-12-14,False,Bantamweight,MALE,3,S-DEC,3,0,4,4,0,13,0,1,0,4,Orthodox,170.18,175.26,31,20,0,1,6,2,15,0,6,0,7,Orthodox,170.18,177.80,31,20


# 3. Data Validation

Now we have an accurate and informative 6000+ fight dataset with to train help train our model. While all of this looks good, let's check for some basic inconsistencies in the data. The first one I want to check is the age column. I want to make sure that we are seeing the age of the fighter at the time of the fight and not the same age all the way down. To check this, I will look at all of Dustin Poirier's UFC fights and see how he's aged.

In [18]:
# Age Check

mask = (df['RedFighter'] == 'Dustin Poirier') | (df['BlueFighter'] == 'Dustin Poirier')
df[mask][['RedFighter', 'RedAge', 'Date', 'BlueFighter', 'BlueAge']]

,RedFighter,RedAge,Date,BlueFighter,BlueAge
291,Islam Makhachev,32,2024-06-01,Dustin Poirier,35
417,Dustin Poirier,35,2024-03-09,Benoit Saint Denis,28
706,Dustin Poirier,34,2023-07-29,Justin Gaethje,34
1068,Dustin Poirier,33,2022-11-12,Michael Chandler,36
1536,Charles Oliveira,32,2021-12-11,Dustin Poirier,32
1758,Dustin Poirier,32,2021-07-10,Conor McGregor,32
1983,Dustin Poirier,32,2021-01-23,Conor McGregor,32
2285,Dustin Poirier,31,2020-06-27,Dan Hooker,30
2622,Khabib Nurmagomedov,30,2019-09-07,Dustin Poirier,30
2838,Max Holloway,27,2019-04-13,Dustin Poirier,30


Age came out perfectly! We can see Dustin from age 21 in his 2011 debut to age 35 in the most recent fight in this dataset. Now I want to make sure that the win streaks work as intended. I'll try this out with a different fighter this time, a fighter I know put together a few win streaks but also lost.

In [19]:
# Win Streak Check

mask = (df['RedFighter'] == 'Justin Gaethje') | (df['BlueFighter'] == 'Justin Gaethje')
df[mask][['RedFighter', 'RedCurrentWinStreak', 'RedLongestWinStreak', 'RedWinner', 'Date', 'BlueFighter', 'BlueCurrentWinStreak', 'BlueLongestWinStreak']]

,RedFighter,RedCurrentWinStreak,RedLongestWinStreak,RedWinner,Date,BlueFighter,BlueCurrentWinStreak,BlueLongestWinStreak
355,Justin Gaethje,2,4,False,2024-04-13,Max Holloway,2,13
706,Dustin Poirier,1,5,False,2023-07-29,Justin Gaethje,1,4
908,Justin Gaethje,0,4,True,2023-03-18,Rafael Fiziev,6,6
1339,Charles Oliveira,10,10,True,2022-05-07,Justin Gaethje,1,4
1589,Justin Gaethje,0,4,True,2021-11-06,Michael Chandler,0,3
2098,Khabib Nurmagomedov,12,12,True,2020-10-24,Justin Gaethje,4,4
2369,Tony Ferguson,12,12,False,2020-05-09,Justin Gaethje,3,3
2611,Donald Cerrone,0,8,False,2019-09-14,Justin Gaethje,2,2
2851,Edson Barboza,1,4,False,2019-03-30,Justin Gaethje,1,1
3147,Justin Gaethje,0,1,True,2018-08-25,James Vick,4,5


As we can see, Justin Gaethje wins his first few fights to build his current and longest streak, but then loses to Khabib Nurmagomedov to reset it. He had built back up a 2 fight winning streak at the time of his Max Holloway fight, but the longest streak remained unaffected.

I want to now look at ranking information. I'll look at this using Israel Adesanya -- a fighter who fought through the rankings and became middleweight champion, but also took a light-heavyweight fight during that reign. We want to see Israel Adesanya's rank rise to champion but be reset to unranked during the Jan Blachowicz fight, going back to champion until some variance around the Sean Strickland and Alex Pereira fights. While we're here, we can also look at the title fight indicator. Let's see what this looks like.

In [20]:
# Rankings Check

mask = (df['RedFighter'] == 'Israel Adesanya') | (df['BlueFighter'] == 'Israel Adesanya')
df[mask][['TitleBout', 'RedFighter', 'RMatchWCRank', 'RedWinner', 'Date', 'WeightClass', 'BlueFighter', 'BMatchWCRank']]

,TitleBout,RedFighter,RMatchWCRank,RedWinner,Date,WeightClass,BlueFighter,BMatchWCRank
171,True,Dricus Du Plessis,0,True,2024-08-17,Middleweight,Israel Adesanya,2
633,True,Israel Adesanya,0,False,2023-09-09,Middleweight,Sean Strickland,5
885,True,Alex Pereira,0,False,2023-04-08,Middleweight,Israel Adesanya,1
1066,True,Israel Adesanya,0,False,2022-11-12,Middleweight,Alex Pereira,4
1256,True,Israel Adesanya,0,True,2022-07-02,Middleweight,Jared Cannonier,2
1475,True,Israel Adesanya,0,True,2022-02-12,Middleweight,Robert Whittaker,1
1794,True,Israel Adesanya,0,True,2021-06-12,Middleweight,Marvin Vettori,3
1930,True,Jan Blachowicz,0,True,2021-03-06,Light Heavyweight,Israel Adesanya,20
2143,True,Israel Adesanya,0,True,2020-09-26,Middleweight,Paulo Costa,2
2385,True,Israel Adesanya,0,True,2020-03-07,Middleweight,Yoel Romero,3


At first glance this may look weird, but this data is actually correct. We can see that in all of the title bouts, the red fighter's rank is 0.0, indicating championship. Israel Adesanya fights to his eventual championship, defends twice, fights the light-heavyweight champion and loses, then moves down to middleweight for a few more championship fights where he was or wasn't the champion.

The final thing I want to check here is wins by submission or knockout. Fans all know that Charles Oliveira is the best fighter to test this with since consistently finishes fights by both method.

In [21]:
# Finish Check

mask = (df['RedFighter'] == 'Charles Oliveira') | (df['BlueFighter'] == 'Charles Oliveira')
df[mask][['Finish', 'RedWinner', 'RedFighter', 'RedWinsByKO', 'RedWinsBySubmission', 'Date', 'BlueFighter','BlueWinsByKO', 'BlueWinsBySubmission']]

,Finish,RedWinner,RedFighter,RedWinsByKO,RedWinsBySubmission,Date,BlueFighter,BlueWinsByKO,BlueWinsBySubmission
41,U-DEC,True,Charles Oliveira,4,16,2024-11-16,Michael Chandler,3,1
356,S-DEC,False,Charles Oliveira,4,16,2024-04-13,Arman Tsarukyan,4,0
793,KO/TKO,True,Charles Oliveira,3,16,2023-06-10,Beneil Dariush,3,5
1102,SUB,False,Charles Oliveira,3,16,2022-10-22,Islam Makhachev,2,5
1339,SUB,True,Charles Oliveira,3,15,2022-05-07,Justin Gaethje,5,0
1536,SUB,True,Charles Oliveira,3,14,2021-12-11,Dustin Poirier,11,3
1832,KO/TKO,True,Charles Oliveira,2,14,2021-05-15,Michael Chandler,2,1
2029,U-DEC,False,Tony Ferguson,0,0,2020-12-12,Charles Oliveira,0,0
2374,SUB,False,Kevin Lee,3,4,2020-03-14,Charles Oliveira,2,13
2505,KO/TKO,True,Charles Oliveira,1,13,2019-11-16,Jared Gordon,1,0


Perfect. We can see Charles Oliveira's submission and knockout totals climb as he wins his fights. This database looks ready to train with.

In [22]:
df.to_csv('../data/df.csv', index = False)